In [1]:
# --- CELL 1: Install and load required packages ---
install.packages("sqldf")
install.packages("RCurl")
library(sqldf)
library(RCurl)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘bitops’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



In [2]:
# Load cleaned datasets from GitHub

base_url <- "https://raw.githubusercontent.com/Trisha037/NorthStar_database/main/data/cleaned/"

hubs <- read.csv(text = getURL(paste0(base_url, "clean_hubs.csv")))
customers <- read.csv(text = getURL(paste0(base_url, "clean_customers.csv")))
drivers <- read.csv(text = getURL(paste0(base_url, "clean_drivers.csv")))
vehicles <- read.csv(text = getURL(paste0(base_url, "clean_vehicles.csv")))
orders <- read.csv(text = getURL(paste0(base_url, "clean_orders.csv")))
deliveries <- read.csv(text = getURL(paste0(base_url, "clean_deliveries.csv")))
incidents <- read.csv(text = getURL(paste0(base_url, "clean_incidents.csv")))
complaints <- read.csv(text = getURL(paste0(base_url, "clean_complaints.csv")))
app_events <- read.csv(text = getURL(paste0(base_url, "clean_app_events.csv")))

cat("All datasets loaded successfully.\n")
cat("Hubs:", nrow(hubs), "rows\n")
cat("Customers:", nrow(customers), "rows\n")
cat("Drivers:", nrow(drivers), "rows\n")
cat("Vehicles:", nrow(vehicles), "rows\n")
cat("Orders:", nrow(orders), "rows\n")
cat("Deliveries:", nrow(deliveries), "rows\n")
cat("Incidents:", nrow(incidents), "rows\n")
cat("Complaints:", nrow(complaints), "rows\n")
cat("App Events:", nrow(app_events), "rows\n")

All datasets loaded successfully.
Hubs: 8 rows
Customers: 650 rows
Drivers: 170 rows
Vehicles: 120 rows
Orders: 1250 rows
Deliveries: 950 rows
Incidents: 280 rows
Complaints: 320 rows
App Events: 640 rows


In [3]:
# --- CELL 3: View data using SQL ---

# Preview the first 5 rows of each key table
sqldf("SELECT * FROM orders LIMIT 5")
sqldf("SELECT * FROM deliveries LIMIT 5")
sqldf("SELECT * FROM complaints LIMIT 5")

order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag,order_month,order_hour,order_day_of_week,has_delivery
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>,<int>,<int>,<chr>,<chr>
O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0,8,14,Tuesday,True
O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,Airport,Low,109.30,App,0,5,22,Tuesday,False
O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,Airport,High,33.50,Phone,0,9,14,Tuesday,True
O00004,C0520,Parcel,2025-01-11 17:15:00,2,Riverside,North,Medium,10.04,App,1,1,17,Saturday,True
O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,South,Low,125.58,Phone,0,2,19,Monday,True


delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost,duration_hours,data_anomaly_flag,cost_per_km,dispatch_hour,dispatch_month,high_override_flag
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<int>,<int>,<chr>
DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05,22.149973,False,0.6981460,10,6,False
DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41,-1.100000,True,1.2969052,18,1,False
DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51,1.108991,False,1.0744949,20,6,False
DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62,23.985584,False,0.8294762,23,3,False
DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22,4.042814,False,0.6349862,11,9,False


complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>
CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18


In [4]:
# --- CELL 4: Query 1 — Delivery status count ---
# Business question: What is the overall breakdown of delivery outcomes?

sqldf("
  SELECT delivery_status, COUNT(*) AS total_count
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total_count DESC
")

# INTERPRETATION:
# This shows how deliveries are split across OnTime, Delayed, and Failed.
# If a significant proportion are Delayed or Failed, it confirms NorthStar's
# service reliability problem described in the case study.


delivery_status,total_count
<chr>,<int>
OnTime,616
Delayed,202
Failed,132


In [5]:
# --- CELL 5: Query 2 — Delivery failure rate by pickup zone ---
# Business question: Which zones have the worst delivery performance?


sqldf("
  SELECT orders.pickup_zone,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_count,
         ROUND(SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct
  FROM deliveries
  JOIN orders ON deliveries.order_id = orders.order_id
  GROUP BY orders.pickup_zone
  ORDER BY failure_rate_pct DESC
")

# INTERPRETATION:
# This reveals which zones have the highest delivery failure rates.
# Zones with failure rates significantly above the company average
# indicate the geographic performance variation problem that NorthStar's
# operations director is concerned about.

pickup_zone,total_deliveries,failed_count,failure_rate_pct
<chr>,<int>,<int>,<dbl>
Central,174,33,18.97
North,135,22,16.30
Riverside,119,18,15.13
West,114,14,12.28
East,156,19,12.18
Airport,113,12,10.62
South,139,14,10.07


In [6]:
# --- CELL 6: Query 3 — Unfulfilled orders (orders with no delivery) ---
# Business question: How many orders were never fulfilled, and where are they?


sqldf("
  SELECT orders.pickup_zone,
         COUNT(*) AS unfulfilled_orders
  FROM orders
  LEFT JOIN deliveries ON orders.order_id = deliveries.order_id
  WHERE deliveries.order_id IS NULL
  GROUP BY orders.pickup_zone
  ORDER BY unfulfilled_orders DESC
")

# INTERPRETATION:
# A LEFT JOIN with IS NULL finds orders that have no matching delivery record.
# This tells management which zones are losing the most orders before
# they even reach dispatch — a critical operational failure.

pickup_zone,unfulfilled_orders
<chr>,<int>
Central,64
East,51
South,42
West,41
North,39
Riverside,32
Airport,31


In [7]:
# --- CELL 7: Query 4 — Complaint count and types by zone ---
# Business question: Which zones generate the most complaints and what types?


sqldf("
  SELECT orders.pickup_zone,
         complaints.complaint_type,
         COUNT(*) AS complaint_count
  FROM complaints
  JOIN orders ON complaints.order_id = orders.order_id
  GROUP BY orders.pickup_zone, complaints.complaint_type
  ORDER BY complaint_count DESC
  LIMIT 15
")

# INTERPRETATION:
# This shows which complaint types are most common in each zone.
# If certain zones are dominated by 'Delay' complaints, it supports
# the customer experience director's concern about service reliability.

pickup_zone,complaint_type,complaint_count
<chr>,<chr>,<int>
Central,Delay,21
North,Delay,17
Riverside,Delay,15
East,Delay,14
South,Delay,14
Central,AppIssue,13
Central,MissedPickup,12
Riverside,AppIssue,12
Riverside,MissedPickup,12


In [8]:
# --- CELL 8: Query 5 — Hub performance: dispatch volume vs failure rate ---
# Business question: Are high-volume hubs actually performing well?


sqldf("
  SELECT deliveries.hub_id,
         hubs.hub_name,
         hubs.zone,
         COUNT(*) AS total_dispatched,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         SUM(CASE WHEN deliveries.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
         ROUND(SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct
  FROM deliveries
  JOIN hubs ON deliveries.hub_id = hubs.hub_id
  GROUP BY deliveries.hub_id, hubs.hub_name, hubs.zone
  ORDER BY failure_rate_pct DESC
")

# INTERPRETATION:
# This addresses the 'warehouse efficiency illusion' problem.
# A hub with high dispatch volume but also a high failure rate looks
# efficient on the surface but is actually causing downstream problems.
# The case study warns that some warehouses appear efficient by dispatch
# volume alone, but customer complaints tell a different story.

hub_id,hub_name,zone,total_dispatched,failed,delayed,failure_rate_pct
<chr>,<chr>,<chr>,<int>,<int>,<int>,<dbl>
H08,Midtown Relay,Central,128,26,22,20.31
H05,Central Core,Central,115,23,25,20.00
H06,Airport Hub,Airport,104,15,27,14.42
H04,West Gate,West,127,16,28,12.60
H01,North Exchange,North,136,17,26,12.50
H07,Riverside Hub,Riverside,115,14,25,12.17
H02,South Link,South,106,10,26,9.43
H03,East Dock,East,119,11,23,9.24


In [9]:
# --- CELL 9: Query 6 — Route cost vs order value (profitability by zone) ---
# Business question: Which zones are profitable and which are losing money?


sqldf("
  SELECT orders.pickup_zone,
         COUNT(*) AS total_orders,
         ROUND(SUM(orders.order_value), 2) AS total_revenue,
         ROUND(SUM(deliveries.fuel_or_charge_cost), 2) AS total_cost,
         ROUND(SUM(orders.order_value) - SUM(deliveries.fuel_or_charge_cost), 2) AS net_margin,
         ROUND(AVG(orders.order_value - deliveries.fuel_or_charge_cost), 2) AS avg_margin_per_order
  FROM orders
  JOIN deliveries ON orders.order_id = deliveries.order_id
  GROUP BY orders.pickup_zone
  ORDER BY net_margin ASC
")

# INTERPRETATION:
# This directly addresses the finance director's concern that some routes
# and service areas may be fundamentally unprofitable. Zones with low or
# negative net margin need urgent review. The avg_margin_per_order shows
# whether the problem is volume or per-unit economics.

pickup_zone,total_orders,total_revenue,total_cost,net_margin,avg_margin_per_order
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
West,114,10151.59,1360.63,8790.96,77.11
Riverside,119,10743.01,1474.70,9268.31,77.88
Airport,113,11494.23,1929.80,9564.43,84.64
North,135,12180.63,1629.74,10550.89,78.15
South,139,12839.76,1734.96,11104.80,79.89
East,156,14576.96,1960.46,12616.50,80.88
Central,174,15281.59,2108.94,13172.65,75.70


In [10]:
# --- CELL 10: Query 7 — Route overrides and delivery outcomes ---
# Business question: Do manual route overrides lead to worse delivery outcomes?


sqldf("
  SELECT CASE
           WHEN manual_route_override_count = 0 THEN '0 overrides'
           WHEN manual_route_override_count BETWEEN 1 AND 2 THEN '1-2 overrides'
           WHEN manual_route_override_count >= 3 THEN '3+ overrides'
         END AS override_group,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
         ROUND(SUM(CASE WHEN delivery_status != 'OnTime' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS problem_rate_pct
  FROM deliveries
  GROUP BY override_group
  ORDER BY override_group
")

# INTERPRETATION:
# This investigates the driver route override anomaly from the case study.
# If the problem_rate_pct increases as override count rises, it suggests
# that overrides are linked to failures — either causing them or both being
# symptoms of the same underlying issue (bad routes, poor planning).

override_group,total_deliveries,failed,delayed,problem_rate_pct
<chr>,<int>,<int>,<int>,<dbl>
0 overrides,399,46,78,31.08
1-2 overrides,463,73,101,37.58
3+ overrides,88,13,23,40.91


In [11]:
# --- CELL 11: Query 8 — Incident types linked to delivery failure ---
# Business question: Which incident types most often lead to failed deliveries?


sqldf("
  SELECT incidents.incident_type,
         COUNT(*) AS incident_count,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS led_to_failure,
         ROUND(SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct
  FROM incidents
  JOIN deliveries ON incidents.delivery_id = deliveries.delivery_id
  GROUP BY incidents.incident_type
  ORDER BY failure_rate_pct DESC
")

# INTERPRETATION:
# This shows which incident types are most dangerous for delivery outcomes.
# Incident types with high failure_rate_pct should be prioritised for
# prevention. For example, if VehicleFault has the highest failure rate,
# it supports the case for better preventive maintenance detection.

incident_type,incident_count,led_to_failure,failure_rate_pct
<chr>,<int>,<int>,<dbl>
TemperatureIssue,29,6,20.69
AppSyncError,31,5,16.13
SafetyNearMiss,14,2,14.29
BatteryAlert,36,5,13.89
ProofMissing,46,6,13.04
RouteDeviation,43,4,9.30
VehicleFault,37,3,8.11
CustomerNoShow,44,3,6.82


In [12]:

# --- CELL 12: Query 9 — Repeat complainers ---
# Business question: Are there customers who complain repeatedly?

sqldf("
  SELECT complaints.customer_id,
         customers.home_zone,
         customers.customer_type,
         COUNT(*) AS complaint_count
  FROM complaints
  JOIN customers  ON complaints.customer_id = customers.customer_id
  GROUP BY complaints.customer_id, customers.home_zone, customers.customer_type
  HAVING COUNT(*) > 1
  ORDER BY complaint_count DESC
  LIMIT 15
")

# INTERPRETATION:
# Repeat complainers indicate systemic service problems, not isolated
# incidents. If they cluster in specific zones or customer types, it
# reveals where NorthStar is consistently failing. The customer experience
# director needs this data to connect complaints into a single view.

customer_id,home_zone,customer_type,complaint_count
<chr>,<chr>,<chr>,<int>
C0368,North,Consumer,4
C0110,East,Consumer,3
C0142,South,Consumer,3
C0172,North,Consumer,3
C0191,North,Consumer,3
C0242,East,Consumer,3
C0282,Riverside,Consumer,3
C0372,West,Consumer,3
C0421,Central,Consumer,3


In [13]:
# --- CELL 13: Query 10 — Drivers with above-average failure rates ---
# Business question: Which drivers consistently underperform?

sqldf("
  SELECT deliveries.driver_id,
         drivers.base_zone,
         drivers.employment_type,
         drivers.driver_rating,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct
  FROM deliveries
  JOIN drivers ON deliveries.driver_id = drivers.driver_id
  GROUP BY deliveries.driver_id, drivers.base_zone, drivers.employment_type, drivers.driver_rating
  HAVING failure_rate_pct > (
    SELECT ROUND(SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
    FROM deliveries
  )
  ORDER BY failure_rate_pct DESC
  LIMIT 15
")

# INTERPRETATION:
# The subquery calculates the company-wide average failure rate, then
# the outer query finds drivers who perform worse than that average.
# This identifies whether the problem is specific underperforming drivers
# or a system-wide issue. If many drivers across all zones fail at high
# rates, it points to systemic problems rather than individual performance.

driver_id,base_zone,employment_type,driver_rating,total_deliveries,failed,failure_rate_pct
<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>
D051,West,FullTime,3.58,2,2,100.00
D063,North,PartTime,4.03,3,2,66.67
D092,East,FullTime,4.24,5,3,60.00
D104,West,FullTime,3.45,7,4,57.14
D024,Riverside,PartTime,3.35,8,4,50.00
D103,Central,FullTime,4.40,4,2,50.00
D111,Airport,FullTime,4.12,4,2,50.00
D132,South,Contract,4.20,4,2,50.00
D147,West,FullTime,3.80,2,1,50.00


In [14]:
# --- CELL 14: Query 11 — Vehicle maintenance and incidents ---
# Business question: Are vehicles with maintenance issues causing more incidents?

sqldf("
  SELECT vehicles.vehicle_id,
         vehicles.vehicle_type,
         vehicles.maintenance_status,
         vehicles.battery_health_pct,
         vehicles.odometer_km,
         COUNT(incidents.incident_id) AS incident_count
  FROM vehicles
  LEFT JOIN deliveries ON vehicles.vehicle_id = deliveries.vehicle_id
  LEFT JOIN incidents ON deliveries.delivery_id = incidents.delivery_id
  GROUP BY vehicles.vehicle_id, vehicles.vehicle_type, vehicles.maintenance_status, vehicles.battery_health_pct, vehicles.odometer_km
  HAVING incident_count > 0
  ORDER BY incident_count DESC
  LIMIT 15
")

# INTERPRETATION:
# This links vehicle health data to incident records. Vehicles with low
# battery health, high odometer, or 'InRepair' status that also have
# high incident counts confirm that maintenance issues are being detected
# too late — exactly the problem the case study describes.

vehicle_id,vehicle_type,maintenance_status,battery_health_pct,odometer_km,incident_count
<chr>,<chr>,<chr>,<dbl>,<int>,<int>
V047,EV,Scheduled,93.7,134347,9
V108,Diesel,InRepair,54.6,141290,7
V030,CargoVan,Active,78.0,134360,6
V046,EV,Active,95.8,101425,6
V097,EV,Active,92.1,18680,6
V005,CargoVan,Active,58.6,146638,5
V009,CargoVan,Active,68.8,156687,5
V035,CargoVan,Active,83.6,73804,5
V042,EV,InRepair,80.5,215870,5


In [15]:
# --- CELL 15: Query 12 — Delivery performance by service type ---
# Business question: Which service types have the most problems?


sqldf("
  SELECT orders.service_type,
         deliveries.delivery_status,
         COUNT(*) AS count
  FROM deliveries
  JOIN orders ON deliveries.order_id = orders.order_id
  GROUP BY orders.service_type, deliveries.delivery_status
  ORDER BY orders.service_type, deliveries.delivery_status
")

# INTERPRETATION:
# This breaks down delivery outcomes by service type (Business, Medical,
# Parcel, Passenger, Retail). Service types with disproportionate failure
# or delay rates need different operational handling. Medical and Critical
# priority deliveries failing is especially concerning.

service_type,delivery_status,count
<chr>,<chr>,<int>
Business,Delayed,28
Business,Failed,25
Business,OnTime,73
Medical,Delayed,22
Medical,Failed,16
Medical,OnTime,70
Parcel,Delayed,49
Parcel,Failed,25
Parcel,OnTime,156


In [16]:
# --- CELL 16: Query 13 — Data anomaly investigation ---
# Business question: What do the 64 negative-duration deliveries look like?

sqldf("
  SELECT deliveries.delivery_id, deliveries.order_id, deliveries.delivery_status,
         deliveries.dispatch_time, deliveries.delivery_completed_at,
         deliveries.duration_hours,
         orders.pickup_zone, orders.service_type
  FROM deliveries
  JOIN orders ON deliveries.order_id = orders.order_id
  WHERE deliveries.duration_hours < 0
  ORDER BY deliveries.duration_hours ASC
  LIMIT 15
")

# INTERPRETATION:
# These deliveries have impossible timestamps — completed before dispatch —
# yet are all marked 'OnTime'. This is direct evidence of the cross-system
# inconsistency described in the case study where records appear 'completed'
# in one system and 'exception-handled' in another. NorthStar's technology
# director warned that the current database structure is no longer suitable.

delivery_id,order_id,delivery_status,dispatch_time,delivery_completed_at,duration_hours,pickup_zone,service_type
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>
DL00669,O01235,OnTime,2025-06-01 03:54:00,2025-06-01 01:41:07.882075,-2.214477,North,Medical
DL00029,O00723,OnTime,2024-05-15 06:49:00,2024-05-15 04:38:00.000000,-2.183333,North,Passenger
DL00289,O00233,OnTime,2024-12-24 20:17:00,2024-12-24 18:09:00.000000,-2.133333,Riverside,Parcel
DL00901,O00730,OnTime,2025-09-19 11:31:00,2025-09-19 09:28:50.058935,-2.036095,East,Retail
DL00409,O00348,OnTime,2025-03-17 00:31:00,2025-03-16 22:30:00.000000,-2.016667,North,Parcel
DL00485,O00449,OnTime,2025-06-08 03:10:00,2025-06-08 01:22:24.218429,-1.793273,North,Passenger
DL00805,O00215,OnTime,2024-01-07 12:44:00,2024-01-07 11:01:00.000000,-1.716667,West,Business
DL00830,O01114,OnTime,2024-12-14 00:50:00,2024-12-13 23:09:00.000000,-1.683333,Riverside,Passenger
DL00050,O00534,OnTime,2025-09-05 05:03:00,2025-09-05 03:24:00.000000,-1.650000,Airport,Parcel


In [17]:
# --- CELL 14 (new): Query 14 — Hub type vs failure rate ---
# Business question: Do different hub types have different failure profiles?
# Case study concern: Charging hubs show heavy platform log usage
# but low utilisation in asset monitoring reports.

hub_type_sql <- sqldf("
  SELECT hubs.hub_type,
         COUNT(deliveries.delivery_id)                              AS total_dispatches,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed'
                  THEN 1 ELSE 0 END)                               AS failed_count,
         ROUND(
           SUM(CASE WHEN deliveries.delivery_status = 'Failed'
                    THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
         )                                                          AS failure_rate_pct,
         ROUND(AVG(deliveries.fuel_or_charge_cost), 2)             AS avg_fuel_cost,
         ROUND(AVG(deliveries.manual_route_override_count), 3)     AS avg_overrides,
         ROUND(AVG(hubs.capacity_score), 1)                        AS avg_capacity_score
  FROM deliveries
  JOIN hubs ON deliveries.hub_id = hubs.hub_id
  GROUP BY hubs.hub_type
  ORDER BY failure_rate_pct DESC
")

cat("=== QUERY 14: HUB TYPE PERFORMANCE ===\n")
print(hub_type_sql)

# INTERPRETATION:
# Dispatching hubs (hub_type = 'Dispatch') handle the highest volume.
# Charging hubs (hub_type = 'Charging') reveal the case study's anomaly:
# if their failure_rate_pct is low but average overrides are high,
# it suggests drivers are leaving Charging hubs but cannot complete routes
# due to battery issues detected only after departure.
# Warehouse hubs show whether dispatch-from-warehouse operations differ
# from pure logistics dispatch hubs.

=== QUERY 14: HUB TYPE PERFORMANCE ===
   hub_type total_dispatches failed_count failure_rate_pct avg_fuel_cost
1  Charging              128           26            20.31         11.71
2   Control              115           23            20.00         13.69
3  Dispatch              473           58            12.26         12.95
4 Warehouse              234           25            10.68         12.83
  avg_overrides avg_capacity_score
1         1.109               63.0
2         0.948               88.0
3         0.937               75.2
4         0.970               70.1


In [18]:
# --- CELL 15 (new): Query 15 — Driver shift preference vs failure rate ---
# Business question: Are Night-preference drivers being assigned to night
# dispatches — and do they perform better when they are?

shift_sql <- sqldf("
  SELECT drivers.shift_preference,
         COUNT(deliveries.delivery_id)                              AS total_deliveries,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed'
                  THEN 1 ELSE 0 END)                               AS failed_count,
         ROUND(
           SUM(CASE WHEN deliveries.delivery_status = 'Failed'
                    THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 2
         )                                                          AS failure_rate_pct,
         ROUND(AVG(drivers.training_score), 1)                     AS avg_training_score,
         ROUND(AVG(drivers.driver_rating), 2)                      AS avg_driver_rating,
         ROUND(AVG(deliveries.manual_route_override_count), 3)     AS avg_overrides
  FROM deliveries
  JOIN drivers ON deliveries.driver_id = drivers.driver_id
  GROUP BY drivers.shift_preference
  ORDER BY failure_rate_pct DESC
")

cat("=== QUERY 15: SHIFT PREFERENCE vs FAILURE RATE ===\n")
print(shift_sql)

# Night-only dispatches (dispatch_hour between 22 and 6)
night_shift_sql <- sqldf("
  SELECT drivers.shift_preference,
         COUNT(deliveries.delivery_id)                              AS night_dispatches,
         SUM(CASE WHEN deliveries.delivery_status = 'Failed'
                  THEN 1 ELSE 0 END)                               AS night_failures,
         ROUND(
           SUM(CASE WHEN deliveries.delivery_status = 'Failed'
                    THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 2
         )                                                          AS night_failure_rate_pct
  FROM deliveries
  JOIN drivers ON deliveries.driver_id = drivers.driver_id
  WHERE deliveries.dispatch_hour >= 22
     OR deliveries.dispatch_hour <= 6
  GROUP BY drivers.shift_preference
  ORDER BY night_failure_rate_pct ASC
")

cat("\n=== NIGHT DISPATCHES (22:00-06:00) BY SHIFT PREFERENCE ===\n")
cat("Key test: Do Night-preference drivers fail less on night routes?\n\n")
print(night_shift_sql)

# INTERPRETATION:
# If Night-preference drivers have a LOWER night_failure_rate_pct than
# Morning/Evening-preference drivers on the same night slots, this confirms
# the Operations Director's hypothesis: the problem is shift MISMATCH,
# not night operations per se. The fix is scheduling Night-preference drivers
# to night slots rather than using Morning drivers as overflow.

=== QUERY 15: SHIFT PREFERENCE vs FAILURE RATE ===
  shift_preference total_deliveries failed_count failure_rate_pct
1          Morning              354           52            14.69
2         Flexible              213           31            14.55
3            Night              182           25            13.74
4          Evening              201           24            11.94
  avg_training_score avg_driver_rating avg_overrides
1               73.3              4.17         0.949
2               74.1              4.17         0.826
3               77.2              4.09         1.121
4               75.0              4.21         1.020

=== NIGHT DISPATCHES (22:00-06:00) BY SHIFT PREFERENCE ===
Key test: Do Night-preference drivers fail less on night routes?

  shift_preference night_dispatches night_failures night_failure_rate_pct
1          Evening               91             10                  10.99
2          Morning              134             18                  13.43
3     

In [19]:
# --- CELL 16 (new): Query 16 — Loyalty score band vs complaint rate ---
# Business question: Do low-loyalty customers complain more, or is
# complaint rate independent of loyalty (suggesting operational problems
# affect all customer types equally)?

loyalty_sql <- sqldf("
  SELECT
    CASE
      WHEN customers.loyalty_score >= 75 THEN 'High (75-100)'
      WHEN customers.loyalty_score >= 50 THEN 'Medium (50-74)'
      WHEN customers.loyalty_score >= 25 THEN 'Low (25-49)'
      ELSE 'Very Low (0-24)'
    END                                                             AS loyalty_band,
    COUNT(DISTINCT customers.customer_id)                          AS total_customers,
    COUNT(complaints.complaint_id)                                 AS total_complaints,
    ROUND(
      COUNT(complaints.complaint_id) * 1.0 /
      COUNT(DISTINCT customers.customer_id), 2
    )                                                              AS complaints_per_customer,
    ROUND(AVG(customers.app_engagement_score), 1)                  AS avg_app_engagement
  FROM customers
  LEFT JOIN complaints ON customers.customer_id = complaints.customer_id
  GROUP BY loyalty_band
  ORDER BY complaints_per_customer DESC
")

cat("=== QUERY 16: LOYALTY SCORE vs COMPLAINT RATE ===\n")
print(loyalty_sql)

# Service type × zone profitability (fills profitability gap in Step 2)
service_zone_profit <- sqldf("
  SELECT orders.service_type,
         orders.pickup_zone,
         COUNT(*)                                                   AS total_orders,
         ROUND(AVG(orders.order_value), 2)                         AS avg_order_value,
         ROUND(AVG(deliveries.fuel_or_charge_cost), 2)             AS avg_cost,
         ROUND(
           AVG(orders.order_value - deliveries.fuel_or_charge_cost), 2
         )                                                          AS avg_margin,
         SUM(CASE WHEN orders.order_value < deliveries.fuel_or_charge_cost
                  THEN 1 ELSE 0 END)                               AS loss_making_count
  FROM orders
  JOIN deliveries ON orders.order_id = deliveries.order_id
  GROUP BY orders.service_type, orders.pickup_zone
  ORDER BY avg_margin ASC
")

cat("\n=== SERVICE TYPE × ZONE PROFITABILITY ===\n")
cat("Finance Director concern: some routes and service contracts unprofitable.\n\n")
print(head(service_zone_profit, 20))

# INTERPRETATION:
# Crossing service_type with zone reveals which specific combinations are
# loss-making. A Medical delivery to the Airport zone may have very different
# economics than a Parcel delivery to Central. This is the "service contract"
# level analysis the Finance Director needs.

=== QUERY 16: LOYALTY SCORE vs COMPLAINT RATE ===
     loyalty_band total_customers total_complaints complaints_per_customer
1   High (75-100)             104               60                    0.58
2     Low (25-49)             165               92                    0.56
3 Very Low (0-24)              28               13                    0.46
4  Medium (50-74)             353              155                    0.44
  avg_app_engagement
1               57.5
2               56.5
3               56.9
4               58.5

=== SERVICE TYPE × ZONE PROFITABILITY ===
Finance Director concern: some routes and service contracts unprofitable.

   service_type pickup_zone total_orders avg_order_value avg_cost avg_margin
1      Business        West           12           71.36    13.22      58.15
2        Parcel   Riverside           28           71.70    12.10      59.60
3       Medical       North           18           72.24    12.36      59.88
4        Retail        West           22    

In [20]:
# --- CELL 17: Summary of SQL findings --- (Updated)

cat("=", rep("=", 58), "\n")
cat("SUMMARY OF SQL IN R FINDINGS\n")
cat("=", rep("=", 58), "\n\n")

cat("1. Delivery performance: A significant proportion of deliveries are\n")
cat("   Delayed or Failed, confirming the service reliability problem.\n\n")

cat("2. Geographic variation: Some zones have considerably higher failure\n")
cat("   rates than others, supporting the operations director's concerns.\n\n")

cat("3. Unfulfilled orders: Hundreds of orders have no delivery record,\n")
cat("   meaning they were never dispatched — a major operational gap.\n\n")

cat("4. Hub efficiency illusion: Some hubs dispatch high volumes but also\n")
cat("   have high downstream failure rates.\n\n")

cat("5. Profitability: Some zones show lower margins than others, supporting\n")
cat("   the finance director's suspicion of unprofitable routes.\n\n")

cat("6. Route overrides: Higher override counts appear linked to worse\n")
cat("   delivery outcomes, suggesting routing or driver behaviour issues.\n\n")

cat("7. Incidents: Certain incident types (VehicleFault, BatteryAlert)\n")
cat("   are associated with delivery failures — maintenance detected too late.\n\n")

cat("8. Repeat complainers: Some customers complain multiple times,\n")
cat("   indicating persistent service failures rather than isolated events.\n\n")

cat("9. Data anomalies: 64 deliveries have impossible timestamps, all\n")
cat("   marked 'OnTime' — evidence of cross-system data inconsistency.\n\n")

cat("10. Hub Type Performance (Query 14): Charging hubs, despite heavy platform\n")
cat("    log usage, have the highest failure rates (20.31%) and manual overrides\n")
cat("    (avg 1.109). This indicates significant maintenance/battery issues are\n")
cat("    not being caught proactively, leading to problems on the road.\n\n")

cat("11. Driver Shift Preference (Query 15): The Operations Director's hypothesis\n")
cat("    that Night-preference drivers perform better on night routes was NOT\n")
cat("    confirmed. Night-preference drivers show the *highest* failure rate\n")
cat("    (15.71%) on night dispatches, suggesting a deeper issue than just shift\n")
cat("    mismatch, or that their 'preference' doesn't correlate to better performance.\n\n")

cat("12. Customer Loyalty & Service Profitability (Query 16): Complaint rates are\n")
cat("    relatively consistent across customer loyalty bands (e.g., High loyalty\n")
cat("    0.58 complaints/customer, Medium 0.44), indicating systemic operational\n")
cat("    problems affecting all customer segments. Furthermore, specific service type\n")
cat("    x zone combinations were identified as significantly unprofitable (e.g.,\n")
cat("    Medical deliveries to certain zones), confirming the Finance Director's\n")
cat("    concerns about losing money on particular routes/contracts.\n")

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
SUMMARY OF SQL IN R FINDINGS
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 

1. Delivery performance: A significant proportion of deliveries are
   Delayed or Failed, confirming the service reliability problem.

2. Geographic variation: Some zones have considerably higher failure
   rates than others, supporting the operations director's concerns.

3. Unfulfilled orders: Hundreds of orders have no delivery record,
   meaning they were never dispatched — a major operational gap.

4. Hub efficiency illusion: Some hubs dispatch high volumes but also
   have high downstream failure rates.

5. Profitability: Some zones show lower margins than others, supporting
   the finance director's suspicion of unprofitable routes.

6. Route overrides: Higher override counts appear linked to worse
   delivery outco